Landing → Bronze | Market Domain
Parses FINWIRE* fixed-width text (CMP/SEC/FIN) into one combined wide Delta table + appends DailyMarket.txt across B1/B2/B3. All columns stored as STRING (bronze convention)

In [0]:
%run ../../02_common_utils/operations

In [0]:
spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("USE SCHEMA bronze")
print("Catalog: charles_schwab_retailbrokerage_dev_team_lemma | Schema: bronze")

In [0]:
dbutils.widgets.text("batch_id", "1", "Batch ID")
dbutils.widgets.text("run_id", "unknown", "Run ID")

carried_batch = dbutils.widgets.get("batch_id")
carried_run_id = dbutils.widgets.get("run_id")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'landing_to_bronze_market_All', f'Starting bronze ingestion for Market domain (batch: {carried_batch})')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'RUNNING')

In [0]:
team_name = "team_lemma"

In [0]:
# Landing base — where raw_to_landing wrote parquet files
landing_base = f"/Volumes/charles_schwab_retailbrokerage_dev_{team_name}/landing/PWG"

# Bronze target database
bronze_db = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"


print(f"Landing base : {landing_base}")
print(f"Bronze target: {bronze_db}")

In [0]:
from pyspark.sql.functions import (
    substring as ss, trim, lit, col,
    current_timestamp, length
)



NULL = lit(None).cast("string")   

def parse_finwire(raw_df):
    """
    Input : raw_df with 'value' column (raw fixed-width text lines) + audit cols.
    Output: combined wide DataFrame — all CMP + SEC + FIN columns,
            NULL for fields that do not belong to a record type.
            All columns remain STRING (bronze convention).
    """
    # Drop non-data lines (header / blank / short lines)
    df = raw_df.filter(length(col("value")) >= 18)

    
    cmp = (df.filter(trim(ss(col("value"), 16, 3)) == "CMP").select(
        ss(col("value"),   1, 15).alias("PTS"),
        ss(col("value"),  16,  3).alias("RecType"),
        # CMP fields
        ss(col("value"),  19, 60).alias("CompanyName"),
        ss(col("value"),  79, 10).alias("CIK"),
        ss(col("value"),  89,  4).alias("Status"),
        ss(col("value"),  93,  2).alias("IndustryID"),
        ss(col("value"),  95,  4).alias("SPrating"),
        ss(col("value"),  99,  8).alias("FoundingDate"),
        ss(col("value"), 107, 80).alias("AddrLine1"),
        ss(col("value"), 187, 80).alias("AddrLine2"),
        ss(col("value"), 267, 12).alias("PostalCode"),
        ss(col("value"), 279, 25).alias("City"),
        ss(col("value"), 304, 20).alias("StateProvince"),
        ss(col("value"), 324, 24).alias("Country"),
        ss(col("value"), 348, 46).alias("CEOname"),
        ss(col("value"), 394,150).alias("Description"),
        # SEC-only → NULL
        NULL.alias("Symbol"),      NULL.alias("IssueType"),
        NULL.alias("Name"),        NULL.alias("ExID"),
        NULL.alias("ShOut"),       NULL.alias("FirstTradeDate"),
        NULL.alias("FirstTradeExchg"), NULL.alias("Dividend"),
        NULL.alias("CoNameOrCIK"),
        # FIN-only → NULL
        NULL.alias("Year"),         NULL.alias("Quarter"),
        NULL.alias("QtrStartDate"), NULL.alias("PostingDate"),
        NULL.alias("Revenue"),      NULL.alias("Earnings"),
        NULL.alias("EPS"),          NULL.alias("DilutedEPS"),
        NULL.alias("Margin"),       NULL.alias("Inventory"),
        NULL.alias("Assets"),       NULL.alias("Liabilities"),
        NULL.alias("DilutedShOut"),
        # Audit columns from landing
        col("_source_file"), col("_batch_id").alias("_batch"),
        col("_ingestion_ts"), col("_run_id")
    ))

    # ── SEC ──────────────────────────────────────────────────────────────
    sec = (df.filter(trim(ss(col("value"), 16, 3)) == "SEC").select(
        ss(col("value"),   1, 15).alias("PTS"),
        ss(col("value"),  16,  3).alias("RecType"),
        # CMP-only → NULL
        NULL.alias("CompanyName"), NULL.alias("CIK"),
        ss(col("value"),  40,  4).alias("Status"),  
        NULL.alias("IndustryID"),  NULL.alias("SPrating"),
        NULL.alias("FoundingDate"), NULL.alias("AddrLine1"),
        NULL.alias("AddrLine2"),   NULL.alias("PostalCode"),
        NULL.alias("City"),        NULL.alias("StateProvince"),
        NULL.alias("Country"),     NULL.alias("CEOname"),
        NULL.alias("Description"),
        # SEC fields
        ss(col("value"),  19, 15).alias("Symbol"),
        ss(col("value"),  34,  6).alias("IssueType"),
        ss(col("value"),  44, 70).alias("Name"),
        ss(col("value"), 114,  6).alias("ExID"),
        ss(col("value"), 120, 13).alias("ShOut"),    
        ss(col("value"), 133,  8).alias("FirstTradeDate"),
        ss(col("value"), 141,  8).alias("FirstTradeExchg"),
        ss(col("value"), 149, 12).alias("Dividend"),
        ss(col("value"), 161, 60).alias("CoNameOrCIK"), 
        # FIN-only → NULL
        NULL.alias("Year"),         NULL.alias("Quarter"),
        NULL.alias("QtrStartDate"), NULL.alias("PostingDate"),
        NULL.alias("Revenue"),      NULL.alias("Earnings"),
        NULL.alias("EPS"),          NULL.alias("DilutedEPS"),
        NULL.alias("Margin"),       NULL.alias("Inventory"),
        NULL.alias("Assets"),       NULL.alias("Liabilities"),
        NULL.alias("DilutedShOut"),
        col("_source_file"), col("_batch_id").alias("_batch"),
        col("_ingestion_ts"), col("_run_id")
    ))

    
    fin = (df.filter(trim(ss(col("value"), 16, 3)) == "FIN").select(
        ss(col("value"),   1, 15).alias("PTS"),
        ss(col("value"),  16,  3).alias("RecType"),
        # CMP-only → NULL
        NULL.alias("CompanyName"), NULL.alias("CIK"),
        NULL.alias("Status"),      NULL.alias("IndustryID"),
        NULL.alias("SPrating"),    NULL.alias("FoundingDate"),
        NULL.alias("AddrLine1"),   NULL.alias("AddrLine2"),
        NULL.alias("PostalCode"),  NULL.alias("City"),
        NULL.alias("StateProvince"), NULL.alias("Country"),
        NULL.alias("CEOname"),     NULL.alias("Description"),
        # SEC-only → NULL
        NULL.alias("Symbol"),      NULL.alias("IssueType"),
        NULL.alias("Name"),        NULL.alias("ExID"),
        ss(col("value"), 161, 13).alias("ShOut"),      
        NULL.alias("FirstTradeDate"), NULL.alias("FirstTradeExchg"),
        NULL.alias("Dividend"),
        ss(col("value"), 187, 60).alias("CoNameOrCIK"), 
        # FIN fields
        ss(col("value"),  19,  4).alias("Year"),
        ss(col("value"),  23,  1).alias("Quarter"),
        ss(col("value"),  24,  8).alias("QtrStartDate"),
        ss(col("value"),  32,  8).alias("PostingDate"),
        ss(col("value"),  40, 17).alias("Revenue"),
        ss(col("value"),  57, 17).alias("Earnings"),
        ss(col("value"),  74, 12).alias("EPS"),
        ss(col("value"),  86, 12).alias("DilutedEPS"),
        ss(col("value"),  98, 12).alias("Margin"),
        ss(col("value"), 110, 17).alias("Inventory"),
        ss(col("value"), 127, 17).alias("Assets"),
        ss(col("value"), 144, 17).alias("Liabilities"),
        ss(col("value"), 174, 13).alias("DilutedShOut"),
        col("_source_file"), col("_batch_id").alias("_batch"),
        col("_ingestion_ts"), col("_run_id")
    ))

    return cmp.unionByName(sec).unionByName(fin)

In [0]:

finwire_root = f"{landing_base}/Batch1/"
finwire_dirs = [
    f.path
    for f in dbutils.fs.ls(finwire_root)
    if f.name.startswith("FINWIRE") and "_audit" not in f.name
]
print(f"Found {len(finwire_dirs)} FINWIRE landing folders")


raw_df = spark.read.parquet(*finwire_dirs)
landing_count = raw_df.count()
print(f"Landing rows (all FINWIRE files, incl. headers): {landing_count}")

parsed_df = parse_finwire(raw_df)


parsed_df.groupBy("RecType").count().show()

(parsed_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_db}.finwire"))

bronze_count = spark.table(f"{bronze_db}.finwire").count()
print(f"bronze.finwire total rows: {bronze_count:,}")


In [0]:


from pyspark.sql.functions import lit, col



for batch in ["1", "2", "3"]:
    path = f"{landing_base}/Batch{batch}/dailymarket"
    df   = spark.read.parquet(path)

    if batch == "1":

        df = (df
              .withColumn("DM_ACTION", lit("I"))          
              .withColumn("DM_RECID",  lit(None).cast("string")))

 
    df = df.select(
        "DM_ACTION", "DM_RECID",
        "DM_DATE", "DM_S_SYMB", "DM_CLOSE",
        "DM_HIGH",  "DM_LOW",    "DM_VOL",
        
        col("_batch_id").alias("_batch"),
        "_source_file", "_ingestion_ts", "_run_id"
    )

    source_count = df.count()

    (df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{bronze_db}.dailymarket"))

    target_count = source_count   
    carried_run_id = str(df.select("`_run_id`").first()[0])

    print(f"Batch{batch} → bronze.dailymarket  rows={source_count:,}")

    
    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id=f"Batch{batch}",
        domain="MARKET",
        table_name="dailymarket",
        source_layer="landing",
        target_layer="bronze",
        source_count=source_count,
        target_count=target_count,
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch=f"Batch{batch}",
        layer="bronze",
        table_name="dailymarket",
        operation="APPEND",
        rows_affected=target_count,
    )


total = spark.table(f"{bronze_db}.dailymarket").count()
print(f"\nbronze.dailymarket cumulative total: {total:,}")


In [0]:



finwire_df   = spark.table(f"{bronze_db}.finwire")
source_count = raw_df.count()                          
target_count = finwire_df.count()                      

carried_run_id = str(finwire_df.select("`_run_id`").first()[0])

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="ALL",
    domain="MARKET",
    table_name="finwire",
    source_layer="landing",
    target_layer="bronze",
    source_count=source_count,
    target_count=target_count,
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="ALL",
    layer="bronze",
    table_name="finwire",
    operation="APPEND",
    rows_affected=target_count,
)

log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'landing_to_bronze_market_All', 'Successfully completed bronze ingestion for market domain.')


# dm_table = spark.table(f"{bronze_db}.dailymarket")

# for b in ["1", "2", "3"]:
#     batch_label  = f"Batch{b}"
#     batch_df     = dm_table.filter(col("_batch") == batch_label)
#     batch_count  = batch_df.count()
#     dm_run_id    = str(batch_df.select("`_run_id`").first()[0])

#     log_pipeline_recon(
#         spark=spark,
#         run_id=dm_run_id,
#         batch_id=batch_label,
#         domain="MARKET",
#         table_name="dailymarket",
#         source_layer="landing",
#         target_layer="bronze",
#         source_count=batch_count,
#         target_count=batch_count,
#     )

#     log_audit_event(
#         spark=spark,
#         run_id=dm_run_id,
#         batch=batch_label,
#         layer="bronze",
#         table_name="dailymarket",
#         operation="APPEND",
#         rows_affected=batch_count,
#     )

# print("Operations log written via log_pipeline_recon + log_audit_event")